# Define volume change areas around Jarkus transects for Terschelling

## Imports

In [1]:
%load_ext autoreload
%autoreload 2

import xarray as xr
import numpy as np
import pandas as pd
import pickle

from shapely import LineString, intersection
from coastal_data import CD_geometry

import pdb

/home/bene/PhD-git/envs/universe/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


## Paths

In [2]:
path_to_data_folder = '/media/bene/Seagate/PhD-data/98_paper2_kf_dtm/'
path_input_general = path_to_data_folder + '0_input_general/'
path_target_area = path_to_data_folder + '12_13_Ters_DEM/output/'
path_output = path_to_data_folder + '05_Ters_transect_polys/'

## Data

In [3]:
jarkus = xr.open_dataset(path_input_general + 'transect_red_with_derivatives.nc')

# Reduce to Terschelling, pick a random year
idx_alongshore, = np.where(jarkus.areacode == 4)
jarkus = jarkus.isel(alongshore=idx_alongshore, time=0)

In [4]:
poly_target = pickle.load(open(path_target_area + 'poly_target4326.pkl', 'rb'))

In [5]:
bs = CD_geometry.dist_meter_to_dist_deg(50) # buffer size
transect_polys = {}
for idx_id in range(0, len(jarkus.id)):
    transect = jarkus.isel(alongshore=idx_id)
    transect_line = LineString(zip(transect.lon.values, transect.lat.values))
    transect_poly = transect_line.buffer(bs)
    # Cut out the part in the target area
    transect_poly_red = intersection(transect_poly, poly_target)
    transect_polys[idx_id] = transect_poly_red

In [6]:
pickle.dump(transect_polys, open(path_output + 'transect_polys.pkl', 'wb'))